# LLM-jp-4 33B BaseをGoogle Colabで4bit実行する

2026年8月18日に公開された **`llm-jp/llm-jp-4-33b-base`** を、Google Colab上でbitsandbytes NF4 4bit量子化して動かします。

このNotebookは **Colab Proの24GB級以上のGPUを第一ターゲット**にしています。33B denseモデルなので、無料版T4では通常のTransformers + 4bit実行はかなり厳しいと予想されます。

Baseモデルはpre-training / mid-trainingまでの基盤モデルです。Chat用のThinkingモデルとは違い、基本的には **文章の続きを生成するcompletion model** として扱います。

```text
Google Colab
  ↓
GPU / VRAM確認
  ↓
Google Drive cache
  ↓
LLM-jp-4 33B BaseをNF4 4bitでロード
  ↓
日本語completion
  ↓
簡単なGradio UI
  ↓
GPUメモリ確認
```


In [1]:
# =========================================
# コード1 実行環境の確認
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U transformers accelerate bitsandbytes sentencepiece

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", GPU_NAME)
print("GPU memory: %.2f GB" % GPU_GB)

if GPU_GB < 20:
    print("WARNING: 33B denseの4bit実行にはVRAMがかなり厳しい可能性があります。")
    print("Colab Proの24GB級以上を推奨します。")


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-e194c279-60aa-d0b0-27c4-b332926a5157)
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 200.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 87.1 MB/s eta 0:00:00
PyTorch: 2.11.0+cu128
Transformers: 5.15.0
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 94.97 GB


In [3]:
# =========================================
# コード2 Google DriveとHugging Face cache
# =========================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil

PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LocalLLM')
CACHE_DIR = PROJECT_DIR / 'Program' / 'hf_cache_llmjp4_33b_base'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HUB_CACHE'] = str(CACHE_DIR)

print('CACHE_DIR:', CACHE_DIR)
print('Drive free: %.1f GB' % (shutil.disk_usage(CACHE_DIR).free/1024**3))


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache_llmjp4_33b_base
Drive free: 178.9 GB


- ここが時間かかる

In [4]:
# =========================================
# コード3 LLM-jp-4 33B BaseをNF4 4bitでロード
# =========================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'llm-jp/llm-jp-4-33b-base'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    cache_dir=str(CACHE_DIR),
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)

model.eval()
INPUT_DEVICE = next(model.parameters()).device

print('model loaded:', MODEL_ID)
print('input device:', INPUT_DEVICE)
print('4bit:', getattr(model, 'is_loaded_in_4bit', False))
print('device map:', getattr(model, 'hf_device_map', 'N/A'))
print('GPU allocated: %.2f GB' % (torch.cuda.memory_allocated()/1024**3))
print('GPU reserved : %.2f GB' % (torch.cuda.memory_reserved()/1024**3))


config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/81.6k [00:00<?, ?B/s]

llmjp4_tokenizer.py:   0%|          | 0.00/2.75k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/llm-jp/llm-jp-4-33b-base:
- llmjp4_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.9MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

model loaded: llm-jp/llm-jp-4-33b-base
input device: cuda:0
4bit: True
device map: N/A
GPU allocated: 18.76 GB
GPU reserved : 19.15 GB


In [5]:
# =========================================
# コード4 Baseモデル用completion生成関数
# =========================================
import gc
import torch

@torch.inference_mode()
def complete_text(prompt, max_new_tokens=128, temperature=0.7):
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        add_special_tokens=True,
    ).to(INPUT_DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=True,
        temperature=float(temperature),
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
        eos_token_id=tokenizer.eos_token_id,
    )

    generated = outputs[0, inputs['input_ids'].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    del inputs, outputs, generated
    gc.collect()
    torch.cuda.empty_cache()
    return text


In [7]:
# =========================================
# コード5 日本語completionテスト
# =========================================
prompt = '''人工知能の歴史を振り返ると、近年の大規模言語モデルの発展には'''

print('PROMPT:')
print(prompt)
print('GENERATED:')
print(complete_text(prompt, max_new_tokens=160, temperature=0.7))


PROMPT:
人工知能の歴史を振り返ると、近年の大規模言語モデルの発展には
GENERATED:
驚嘆するばかりです。GPT-3 のようなモデルは、テキスト生成や翻訳、質問応答など、多くのタスクで驚異的なパフォーマンスを発揮しています。これらのモデルがどのようにしてこれほどまでに進化を遂げたのか、その過程を探ることは非常に重要です。

#### 1. 初期の研究と理論
人工知能の研究は 1950 年代から始まり、当初はルールベースのシステムが主流でした。この時代の AI は、人間が定義した規則に従って動作するものでした。しかし、これらのシステムは柔軟性に欠け、複雑なタスクには対応できませんでした。

#### 2. ニューラルネットワークの登場
1980 年代に入ると、ニューラルネットワークが再び注目を浴びるようになりました。特に、バックプロパゲーションアルゴリズムの開発により、多層パーセプトロン（MLP）が実用化されました。これにより、画像認識や音声認識


## Baseモデルについて

`-base`はChat用にSFT/DPOされたモデルではありません。そのため、質問応答UIよりも「入力文章の続きを生成する」使い方が自然です。

Thinkingや指示追従を試したい場合は、別Notebookの **`llm-jp-4-33b-thinking`** を利用してください。


In [8]:
# =========================================
# コード6 Gradio Completion UI
# =========================================
import gradio as gr

print('Gradio:', gr.__version__)

def gr_complete(prompt, max_tokens, temperature):
    if not (prompt or '').strip():
        return '文章を入力してください。'
    try:
        return complete_text(prompt, int(max_tokens), float(temperature))
    except Exception as e:
        import traceback
        traceback.print_exc()
        return f'{type(e).__name__}: {e}'

with gr.Blocks(title='LLM-jp-4 33B Base') as demo:
    gr.Markdown('## LLM-jp-4 33B Base — 4bit Completion')
    prompt_box = gr.Textbox(
        label='Prompt',
        value='人工知能の歴史を振り返ると、近年の大規模言語モデルの発展には',
        lines=6,
    )
    max_box = gr.Slider(32, 512, value=128, step=32, label='max_new_tokens')
    temp_box = gr.Slider(0.1, 1.2, value=0.7, step=0.1, label='temperature')
    run_btn = gr.Button('Generate', variant='primary')
    output_box = gr.Textbox(label='Continuation', lines=12)
    run_btn.click(gr_complete, inputs=[prompt_box, max_box, temp_box], outputs=output_box, concurrency_limit=1)

demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=True, debug=False, show_error=True)


Gradio: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57129b8a997bce0abb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
# =========================================
# コード7 GPUメモリ確認
# =========================================
import gc, torch
gc.collect(); torch.cuda.empty_cache()

print('GPU allocated: %.2f GB' % (torch.cuda.memory_allocated()/1024**3))
print('GPU reserved : %.2f GB' % (torch.cuda.memory_reserved()/1024**3))
free, total = torch.cuda.mem_get_info()
print('GPU free      : %.2f GB' % (free/1024**3))
print('GPU total     : %.2f GB' % (total/1024**3))


GPU allocated: 18.77 GB
GPU reserved : 19.14 GB
GPU free      : 75.17 GB
GPU total     : 94.97 GB


## 実験時に記録しておく項目

```text
GPU名 / VRAM
モデルロード成功・失敗
4bitロード直後のGPU allocated
日本語completionの結果
Gradio動作
最終GPU free memory
```

特に無料版T4でも試す場合は、**ロードがどの段階で失敗するか**を記録すると、Colab Proとの比較記事に利用できます。
